# 第二周 · 第 3-4 天：财务数据获取

> 目标：掌握用 akshare 获取个股财务报表（利润表、资产负债表、现金流量表），理解各项财务指标的含义

---

## 1. 三大财务报表概述

| 报表 | 反映内容 | 核心指标 |
|------|----------|----------|
| 利润表 | 盈利情况 | 营收、净利润、毛利率、净利率 |
| 资产负债表 | 财务状况 | 总资产、总负债、净资产 |
| 现金流量表 | 现金流动 | 经营现金流、投资现金流、筹资现金流 |

### 财务报表之间的关系

```
净利润 = 营收 - 成本 - 费用 - 税
净资产 = 总资产 - 总负债（资产负债表）
经营现金流 ≠ 净利润（权责发生制 vs 收付实现制）
```

---

## 2. 获取利润表

```python
import akshare as ak
import pandas as pd

# 股票代码
symbol = "000001"  # 平安银行

# 获取利润表
df_income = ak.stock_financial_analysis_indicator(symbol=symbol)
print(f"列名: {list(df_income.columns)}")
print(f"数据形状: {df_income.shape}")
```

### 关键利润表指标

| 指标 | 含义 | 优质公司标准 |
|------|------|-------------|
| 营业总收入 | 主营业务 + 其他业务 | 稳定增长 |
| 净利润 | 扣除所有成本后的利润 | > 0 且稳定 |
| 营业总成本 | 主营业务成本 + 费用 | 成本控制良好 |
| 销售毛利率 | (营收-成本)/营收 | > 30% |
| 销售净利率 | 净利润/营收 | > 10% |

---

## 3. 获取资产负债表

```python
# 获取资产负债表（不同接口）
df_balance = ak.stock_financial_report_sina(symbol=symbol, symbol_name="balance")
print(f"资产负债表列名: {list(df_balance.columns)}")
print(df_balance.head())
```

### 关键资产负债表指标

| 指标 | 含义 | 优质公司标准 |
|------|------|-------------|
| 总资产 | 公司拥有资源的总价值 | 稳定增长 |
| 总负债 | 公司需要偿还的债务 | 负债率合理 |
| 净资产 | 总资产 - 总负债 | 稳定增长 |
| 资产负债率 | 总负债/总资产 | < 60% |
| 流动比率 | 流动资产/流动负债 | > 1.5 |

---

## 4. 获取现金流量表

```python
# 获取现金流量表
df_cashflow = ak.stock_financial_report_sina(symbol=symbol, symbol_name="cashflow")
print(f"现金流量表列名: {list(df_cashflow.columns)}")
print(df_cashflow.head())
```

### 关键现金流量表指标

| 指标 | 含义 | 优质公司标准 |
|------|------|-------------|
| 经营现金流 | 主业产生的现金流 | > 0 且稳定 |
| 投资现金流 | 投资支出/收入 | 合理范围 |
| 筹资现金流 | 融资/分红 | 反映融资需求 |

### 现金流分析

```python
# 经营现金流与净利润对比
# 理想情况：经营现金流 > 净利润（利润质量高）
# 危险信号：经营现金流 < 0 或持续低于净利润
```

---

## 5. 财务指标综合分析

```python
import akshare as ak
import pandas as pd

# 获取财务指标
symbol = "000001"
df = ak.stock_financial_analysis_indicator(symbol=symbol)

# 查看关键指标（按日期排序，取最新）
df = df.sort_values('日期', ascending=False)

# 关键财务指标
key_metrics = [
    '日期',
    '净资产收益率(%)',        # ROE
    '销售毛利率(%)',         # 毛利率
    '销售净利率(%)',         # 净利率
    '资产负债率(%)',         # 负债率
    '存货周转率(次)',         # 存货周转
    '应收账款周转率(次)',     # 应收账款周转
]

# 检查哪些列存在
available_cols = [col for col in key_metrics if col in df.columns]
print(f"可用指标: {available_cols}")
print(df[available_cols].head(10))
```

---

## 6. 多股票财务数据对比

```python
# 选几只银行股对比财务指标
stock_list = [
    ("000001", "平安银行"),
    ("600000", "浦发银行"),
    ("600036", "招商银行"),
    ("601166", "兴业银行"),
]

results = []
for code, name in stock_list:
    try:
        df = ak.stock_financial_analysis_indicator(symbol=code)
        df = df.sort_values('日期', ascending=False)

        # 取最新一期数据
        latest = df.iloc[0]

        results.append({
            '股票代码': code,
            '股票名称': name,
            'ROE(%)': latest.get('净资产收益率(%)', None),
            '毛利率(%)': latest.get('销售毛利率(%)', None),
            '净利率(%)': latest.get('销售净利率(%)', None),
            '资产负债率(%)': latest.get('资产负债率(%)', None),
        })
    except Exception as e:
        print(f"{code} 获取失败: {e}")

df_comparison = pd.DataFrame(results)
print("\n银行股财务指标对比:")
print(df_comparison)
```

---

## 7. 财务筛选实战

```python
# 获取全市场股票财务数据，筛选优质标的
# 条件：
# 1. ROE > 10%（盈利能力）
# 2. 毛利率 > 20%（竞争力）
# 3. 资产负债率 < 70%（财务风险）

# 获取所有 A 股实时数据（含财务指标）
df_all = ak.stock_zh_a_spot_em()
print(f"全市场股票数量: {len(df_all)}")

# 筛选条件
df_filtered = df_all[
    (df_all['市盈率-动态'] > 0) &
    (df_all['市盈率-动态'] < 30) &
    (df_all['市净率'] > 1) &
    (df_all['市净率'] < 5) &
    (df_all['换手率'] > 2)
].copy()

# 按市值排序（优先大公司）
df_filtered = df_filtered.sort_values('总市值', ascending=False)

print(f"\n初步筛选后股票数量: {len(df_filtered)}")
print(df_filtered[['代码', '名称', '最新价', '市盈率-动态', '市净率', '换手率', '总市值']].head(30))
```

---

## 8. 财务数据存储

```python
from data.database import StockDataDB

db = StockDataDB()

# 存储财务数据到 MySQL
def save_financial_data(stock_code: str, df: pd.DataFrame, report_type: str):
    """保存财务数据"""
    df_copy = df.copy()
    df_copy['stock_code'] = stock_code
    df_copy['report_type'] = report_type  # income/balance/cashflow

    # 写入数据库
    df_copy.to_sql('stock_financial', engine, if_exists='replace', index=False)
    print(f"✅ {stock_code} {report_type} 数据已保存: {len(df_copy)} 条")

# 示例
save_financial_data("000001", df_income, "indicator")
```

---

## 9. 今日任务清单

- [x] 理解三大财务报表（利润表、资产负债表、现金流量表）
- [x] 掌握用 akshare 获取财务报表数据
- [x] 理解关键财务指标（ROE、毛利率、净利率、资产负债率）
- [x] 实现多股票财务指标对比
- [x] 实现财务数据筛选优质标的
- [x] 将财务数据存入 MySQL

---

## ✅ 今日要点总结

| 报表 | 核心问题 | 关键指标 |
|------|----------|----------|
| 利润表 | 公司赚钱吗？ | 净利润、毛利率、净利率 |
| 资产负债表 | 公司有多少家当？ | 总资产、净资产、负债率 |
| 现金流量表 | 利润是真钱吗？ | 经营现金流 |

| 财务指标 | 优质标准 |
|----------|----------|
| ROE | > 10%，越高越好 |
| 毛利率 | > 20%，越高竞争力越强 |
| 资产负债率 | < 70%，越低越安全 |
| 经营现金流 | > 净利润，利润质量高 |

---

> 完成日期：2026-07-01
> 下一步：第二周 第 5-6 天 - 选股池构建实战（10 只股票的关键财务指标表格）
> 第 7 天：周末输出 - 选股池 + 数据采集代码

In [ ]:
import akshare as ak
import pandas as pd

# 股票代码
symbol = "000001"  # 平安银行

# 获取财务指标
df = ak.stock_financial_analysis_indicator(symbol=symbol)
print(f"列名: {list(df.columns)}")
print(f"数据形状: {df.shape}")
print(f"\n最新数据日期: {df['日期'].max()}")
print(df.tail())

In [ ]:
# 关键财务指标（取最新一期）
df = df.sort_values('日期', ascending=False)

# 尝试获取关键指标
key_metrics = [
    '日期',
    '净资产收益率(%)',        # ROE
    '销售毛利率(%)',         # 毛利率
    '销售净利率(%)',         # 净利率
    '资产负债率(%)',         # 负债率
]

available_cols = [col for col in key_metrics if col in df.columns]
print(f"可用指标: {available_cols}")
print(df[available_cols].head(10))

In [ ]:
# 多股票财务指标对比
stock_list = [
    ("000001", "平安银行"),
    ("600000", "浦发银行"),
    ("600036", "招商银行"),
    ("601166", "兴业银行"),
]

results = []
for code, name in stock_list:
    try:
        df_temp = ak.stock_financial_analysis_indicator(symbol=code)
        df_temp = df_temp.sort_values('日期', ascending=False)
        latest = df_temp.iloc[0]
        results.append({
            '股票代码': code,
            '股票名称': name,
            'ROE(%)': latest.get('净资产收益率(%)', None),
            '毛利率(%)': latest.get('销售毛利率(%)', None),
            '净利率(%)': latest.get('销售净利率(%)', None),
            '资产负债率(%)': latest.get('资产负债率(%)', None),
        })
        print(f"✅ {code} {name} 获取成功")
    except Exception as e:
        print(f"❌ {code} 获取失败: {e}")

df_comparison = pd.DataFrame(results)
print("\n=== 银行股财务指标对比 ===")
print(df_comparison)

In [ ]:
# 获取全市场股票，筛选优质标的
df_all = ak.stock_zh_a_spot_em()
print(f"全市场股票数量: {len(df_all)}")

# 筛选条件
df_filtered = df_all[
    (df_all['市盈率-动态'] > 0) &
    (df_all['市盈率-动态'] < 30) &
    (df_all['市净率'] > 1) &
    (df_all['市净率'] < 5) &
    (df_all['换手率'] > 2)
].copy()

# 按市值排序
df_filtered = df_filtered.sort_values('总市值', ascending=False)

print(f"\n初步筛选后股票数量: {len(df_filtered)}")
print(df_filtered[['代码', '名称', '最新价', '市盈率-动态', '市净率', '换手率', '总市值']].head(30))

In [ ]:
# 尝试获取财务报表（利润表、资产负债表、现金流量表）
# 注意：akshare 提供了多个财务数据接口，不同接口返回的数据结构不同

# 方法1：财务指标综合表
df_indicator = ak.stock_financial_analysis_indicator(symbol="000001")
print("=== 财务指标综合表 ===")
print(f"包含 {len(df_indicator)} 个指标")
print(df_indicator.columns.tolist()[:20])